In [1]:
import gzip
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import gc
import hashlib  # Added for text hashing

# ================= Configuration =================
REVIEW_FILE = "All_Amazon_Review_5.json.gz"
META_FILE = "All_Amazon_Meta.json.gz"
OUTPUT_FILE = "amazon_recent_20m_dataset.csv"

TARGET_SIZE = 20_000_000
MIN_ITEM_COUNT = 5
MIN_SEQ_LEN = 20

In [ ]:
import gzip
import json
from tqdm import tqdm

def load_item_categories(filepath):
    """
    Loads item categories from a double-compressed metadata file using streaming.
    Structure: GZIP -> GZIP -> JSON Lines
    """
    item_map = {}
    print(f"Loading metadata from {filepath} (Double Compression Mode)...")
    
    try:
        with gzip.open(filepath, 'rb') as f_outer:
            
            with gzip.open(f_outer, mode='rt', encoding='utf-8') as f_inner:
                
                for line in tqdm(f_inner, desc="Parsing Metadata"):
                    try:
                        entry = json.loads(line.strip())
                        
                        asin = entry.get('asin')
                        categories = entry.get('category', [])
                        
                        if asin:
                            if categories and len(categories) > 0:
                                item_map[asin] = categories[0]
                            else:
                                item_map[asin] = "Unknown"
                                
                    except json.JSONDecodeError:
                        continue
                    except Exception:
                        continue
                        
    except Exception as e:
        print(f"Error reading metadata: {e}")
        
    print(f"Loaded category information for {len(item_map)} items.")
    return item_map

# 执行函数
item_category_map = load_item_categories(META_FILE)

# 预览前几个
print("\nSample Category Mappings:")
print(list(item_category_map.items())[:5])

Loading metadata from All_Amazon_Meta.json.gz (Double Compression Mode)...


Parsing Metadata: 15023059it [12:26, 20133.37it/s]


Loaded category information for 14741571 items.

Sample Category Mappings:
[('6305121869', 'Clothing, Shoes & Jewelry'), ('6318708057', 'Clothing, Shoes & Jewelry'), ('6342506256', 'Clothing, Shoes & Jewelry'), ('6342509379', 'Clothing, Shoes & Jewelry'), ('6342522081', 'Clothing, Shoes & Jewelry')]


In [ ]:
import gzip
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import gc




# ================= Phase 1: Timestamp Scanning =================
print("Phase 1: Scanning timestamps to find cutoff time...")
all_timestamps = []

# Pass 1: Read only timestamps to determine the cutoff for the most recent data
with gzip.open(REVIEW_FILE, "rt", encoding="utf-8") as f:
    for line in tqdm(f, desc="Reading Timestamps"):
        try:
            # Quick parsing: only extract 'unixReviewTime'
            entry = json.loads(line.strip())
            ts = entry.get('unixReviewTime')
            if ts:
                all_timestamps.append(ts)
        except:
            continue

print(f"Total records found: {len(all_timestamps)}")

if len(all_timestamps) > TARGET_SIZE:
    # Use numpy partition to find the k-th largest element (faster than full sort)
    # We need the timestamp that splits the data into "old" vs "recent 20M"
    cutoff_index = len(all_timestamps) - TARGET_SIZE
    
    partitioned = np.partition(np.array(all_timestamps), cutoff_index)
    cutoff_time = partitioned[cutoff_index]
    
    print(f"Cutoff Timestamp determined: {cutoff_time}")
    print(f"(Keeping data on or after this time)")
else:
    cutoff_time = 0
    print("Dataset size is smaller than target. Keeping all.")

# Free up memory
del all_timestamps
del partitioned
gc.collect()
print("\nPhase 2: Loading data (Timestamp >= Cutoff) with Chunking...")

BATCH_SIZE = 1_000_000
chunks = []
batch_data = []

with gzip.open(REVIEW_FILE, "rt", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading Recent Reviews"):
        try:
            entry = json.loads(line.strip())
            ts = entry.get('unixReviewTime')
            
            # 1. Timestamp Filter
            if ts is None or ts < cutoff_time:
                continue
            
            # 2. Extract Text & Hash
            review_text = entry.get('reviewText', "")
            text_hash = hashlib.md5(review_text.encode('utf-8')).hexdigest()
            rating = entry.get('overall') 
            batch_data.append({
                'user_id_raw': entry.get('reviewerID'),
                'item_id_raw': entry.get('asin'),
                'timestamp': ts,
                'text_hash': text_hash,
                'rating': rating  # 
            })

            if len(batch_data) >= BATCH_SIZE:
                df_chunk = pd.DataFrame(batch_data)
                df_chunk['timestamp'] = df_chunk['timestamp'].astype('int32')
                df_chunk['rating'] = df_chunk['rating'].astype('float32') 
                chunks.append(df_chunk)
                batch_data = []
                gc.collect()
                
        except Exception:
            continue
if batch_data:
    df_chunk = pd.DataFrame(batch_data)
    df_chunk['timestamp'] = df_chunk['timestamp'].astype('int32')
    df_chunk['rating'] = df_chunk['rating'].astype('float32') 
    chunks.append(df_chunk)
    del batch_data
    gc.collect()


print(f"Loaded {len(chunks)} chunks. Concatenating...")

df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

print(f"Original Raw Shape: {df.shape}")

# --- Deduplication ---
print("Applying Deduplication...")
df['review_date'] = pd.to_datetime(df['timestamp'], unit='s').dt.date

initial_len = len(df)
df.drop_duplicates(subset=['user_id_raw', 'review_date', 'text_hash'], keep='first', inplace=True)
print(f"Removed {initial_len - len(df)} duplicate reviews.")

# Clean up
df.drop(columns=['review_date', 'text_hash'], inplace=True)
gc.collect()

# --- Standard Filtering ---
print("Filtering unpopular items...")
item_counts = df['item_id_raw'].value_counts()
valid_items = item_counts[item_counts >= MIN_ITEM_COUNT].index
df = df[df['item_id_raw'].isin(valid_items)].copy()

print("Filtering short sequences...")
user_counts = df['user_id_raw'].value_counts()
valid_users = user_counts[user_counts >= MIN_SEQ_LEN].index
df = df[df['user_id_raw'].isin(valid_users)].copy()

print(f"Final Data Shape: {df.shape}")


Phase 1: Scanning timestamps to find cutoff time...


Reading Timestamps: 157260920it [22:52, 114608.16it/s]


Total records found: 157260920
Cutoff Timestamp determined: 1503273600
(Keeping data on or after this time)

Phase 2: Loading data (Timestamp >= Cutoff) with Chunking...


Loading Recent Reviews: 157260920it [22:03, 118842.79it/s]


Loaded 21 chunks. Concatenating...
Original Raw Shape: (20028692, 5)
Applying Deduplication...
Removed 1706926 duplicate reviews.
Filtering unpopular items...
Filtering short sequences...
Final Data Shape: (2186346, 4)


In [4]:
df.head()

,user_id_raw,item_id_raw,timestamp,rating
26,A2MM9UNRO2H898,B017OBYNZU,1516752000,5.0
101,A2XR8OTPD7KOP4,B017OBSCOS,1504742400,5.0
168,A1NEIH01B5X6QL,B017RXFNKY,1522108800,5.0
188,A149V0099W5Y9O,B017RXFNKY,1516665600,5.0
194,A2WDLP4X3XIIYT,B017RXFNKY,1515888000,4.0


In [5]:
# ================= Phase 3.5: Metadata Loading & Mapping =================
print("\nPhase 3.5: Loading Metadata & Mapping Categories...")

def load_item_categories_streaming(filepath):
    """Stream-load metadata to map ASIN to Category"""
    item_map = {}
    print(f"Loading metadata from {filepath}...")
    try:
        with gzip.open(filepath, 'rb') as f_outer:
            with gzip.open(f_outer, mode='rt', encoding='utf-8') as f_inner:
                for line in tqdm(f_inner, desc="Parsing Meta"):
                    try:
                        entry = json.loads(line.strip())
                        asin = entry.get('asin')
                        categories = entry.get('category', [])
                        
                        if asin:
                            item_map[asin] = categories[0] if categories else "Unknown"
                    except:
                        continue
    except Exception as e:
        print(f"Error reading metadata: {e}")
    return item_map

item_category_map = load_item_categories_streaming(META_FILE)

print("Mapping categories to dataframe...")
df['category'] = df['item_id_raw'].map(item_category_map).fillna('Unknown')

del item_category_map
gc.collect()



Phase 3.5: Loading Metadata & Mapping Categories...
Loading metadata from All_Amazon_Meta.json.gz...


Parsing Meta: 15023059it [11:31, 21734.00it/s]


Mapping categories to dataframe...


9

In [ ]:
# ================= New Feature: Add Product Title =================
import gzip
import json
from tqdm import tqdm

NEW_OUTPUT_FILE = "amazon_recent_20m_dataset_with_title.csv"

def load_item_titles_streaming(filepath):
    title_map = {}
    print(f"Loading titles from {filepath}...")
    try:
        with gzip.open(filepath, 'rb') as f_outer:
            with gzip.open(f_outer, mode='rt', encoding='utf-8') as f_inner:
                for line in tqdm(f_inner, desc="Parsing Titles"):
                    try:
                        entry = json.loads(line.strip())
                        asin = entry.get('asin')
                        title = entry.get('title')
                        
                        if asin and title:
                            title_map[asin] = title
                    except:
                        continue
    except Exception as e:
        print(f"Error reading metadata: {e}")
    
    print(f"Loaded titles for {len(title_map)} items.")
    return title_map

item_title_map = load_item_titles_streaming(META_FILE)
print("Mapping titles to DataFrame...")
df['title'] = df['item_id_raw'].map(item_title_map).fillna('Unknown')


if 'user_id' not in df.columns:
    print("Generating IDs...")
    user_map = {u: i+1 for i, u in enumerate(df['user_id_raw'].unique())}
    item_map = {item: i+1 for i, item in enumerate(df['item_id_raw'].unique())}
    df['user_id'] = df['user_id_raw'].map(user_map)
    df['item_id'] = df['item_id_raw'].map(item_map)

final_cols = ['user_id', 'item_id', 'timestamp', 'category', 'rating', 'title']
final_df_with_title = df[final_cols]

print(f"Saving to {NEW_OUTPUT_FILE}...")
final_df_with_title.to_csv(NEW_OUTPUT_FILE, index=False)
print("Done.")

Loading titles from All_Amazon_Meta.json.gz...


Parsing Titles: 15023059it [10:41, 23414.28it/s]


Loaded titles for 14737206 items.
Mapping titles to DataFrame...
Generating IDs...
Saving to amazon_recent_20m_dataset_with_title.csv...
Done.


In [ ]:

# ================= Phase 4: Final Re-indexing (IDs) =================
print("\nPhase 4: Finalizing IDs and saving...")

# 1. Sort by User and Time
# Adding item_id_raw as secondary sort key for deterministic results
df.sort_values(by=['user_id_raw', 'timestamp', 'item_id_raw'], inplace=True)

# 2. Convert Timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# --- [Modification 2] Generate IDs AFTER all filtering is done ---
print("Generating new User/Item IDs based on filtered data...")
unique_users = df['user_id_raw'].unique()
unique_items = df['item_id_raw'].unique()

user_map = {u: i+1 for i, u in enumerate(unique_users)}
item_map = {item: i+1 for i, item in enumerate(unique_items)}

df['user_id'] = df['user_id_raw'].map(user_map)
df['item_id'] = df['item_id_raw'].map(item_map)

# 3. Select final columns

final_df = df[['user_id', 'item_id', 'timestamp', 'category', 'rating']]

print("\nDataset Statistics:")
print(final_df.info())
print(f"Total Users: {final_df['user_id'].nunique()}")
print(f"Total Items: {final_df['item_id'].nunique()}")
print(f"Total Interactions: {len(final_df)}")

# Save
print(f"\nSaving to {OUTPUT_FILE}...")
final_df.to_csv(OUTPUT_FILE, index=False)
print("Done.")


Phase 4: Finalizing IDs and saving...
Generating new User/Item IDs based on filtered data...

Dataset Statistics:
<class 'pandas.core.frame.DataFrame'>
Index: 2186346 entries, 17649895 to 12923219
Data columns (total 5 columns):
 #   Column     Dtype         
---  ------     -----         
 0   user_id    int64         
 1   item_id    int64         
 2   timestamp  datetime64[ns]
 3   category   object        
 4   rating     float32       
dtypes: datetime64[ns](1), float32(1), int64(2), object(1)
memory usage: 91.7+ MB
None
Total Users: 65262
Total Items: 453454
Total Interactions: 2186346

Saving to amazon_recent_20m_dataset.csv...
Done.
